# Phase 6: Model Training (ViT-B/16)
Bu notebook'ta, bir önceki aşamada hazırladığımız gelişmiş veri ön işleme (Advanced Data Prep + WeightedRandomSampler) pipeline'ını kullanarak ViT-B/16 modelini (Transfer Learning) eğiteceğiz.

## İçerik:
- Modelin Yüklenmesi (timm kütüphanesi ile ViT-B/16)
- Classifier Head'in Değiştirilmesi (102 Sınıf için)
- Optimizer (AdamW), Loss Function (CrossEntropy) ve Scheduler
- Eğitim ve Validasyon Döngüsü (Training Loop)
- Metrikler (Accuracy, F1-Score) ve Early Stopping

In [1]:
import os
import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score
import timm

# Proje kök dizinini sys.path'e dinamik olarak ekliyoruz:
import sys
current_dir = os.getcwd()
if current_dir.endswith('notebooks'):
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir
sys.path.append(project_root)

from dataset import get_dataloaders


c:\Users\Asus\OneDrive\Masaüstü\capstone_final\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch

print("PyTorch Sürümü:", torch.__version__)
print("CUDA Aktif mi?:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Kullanılan Ekran Kartı:", torch.cuda.get_device_name(0))
else:
    print("CUDA aktif değil, CPU kullanılıyor.")


PyTorch Sürümü: 2.5.1+cu121
CUDA Aktif mi?: True
Kullanılan Ekran Kartı: NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
# Config
DATA_DIR = '../data/classification'
BATCH_SIZE = 16  # AMP sayesinde bellekte yer açıldığı için 4 katına çıkardık!
IMG_SIZE = 224
NUM_EPOCHS = 30
LEARNING_RATE = 3e-5

# NVIDIA Ekran Kartı için Özel Optimizasyon (Hızı artırır)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Kullanılan Cihaz: {DEVICE}")


Kullanılan Cihaz: cuda


## 1. Dataloader'ları Başlatma

In [4]:
train_loader, val_loader, test_loader, num_classes = get_dataloaders(
    data_dir=DATA_DIR, 
    img_size=IMG_SIZE, 
    batch_size=BATCH_SIZE
)
print(f"Toplam Sınıf Sayısı: {num_classes}")

c:\Users\Asus\OneDrive\Masaüstü\capstone_final\venv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Toplam Sınıf Sayısı: 102


## 2. ViT-B/16 Modelinin Yüklenmesi (Transfer Learning)
Önceden eğitilmiş (Pre-trained) ağırlıkları kullanıyoruz. Orijinal modelin sınıflandırıcı başlığı (classifier head) ImageNet'teki 1000 sınıfa ayarlıdır. Biz bunu `num_classes=102` olarak değiştiriyoruz.

In [5]:
# 'vit_base_patch16_224' modeli Proposal'da belirtilen modeldir.
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes)
model = model.to(DEVICE)
print("ViT modeli başarıyla yüklendi ve Classifier Head değiştirildi.")

ViT modeli başarıyla yüklendi ve Classifier Head değiştirildi.


## 3. Loss, Optimizer ve Scheduler

In [6]:
# Dengesiz veri problemi için WeightedRandomSampler kullanıyoruz.
# Ancak yine de modelin aşırı güvenini (overconfidence) önlemek ve 
# fine-grained classification performansını artırmak için Label Smoothing ekliyoruz.
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ViT modelleri için standart olan AdamW optimizer kullanılır (Weight Decay regularization için)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Learning Rate Scheduler: Eğitim ilerledikçe LR'yi kademeli olarak düşürür
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

## 4. Eğitim Döngüsü (Training Loop) & Early Stopping
Modeli eğitirken her Epoch'ta Training ve Validation metriklerini (Loss, Accuracy, F1-Score) hesaplayacağız. 
Ayrıca model gelişme göstermezse eğitimi erkenden bitiren (Early Stopping) bir yapı kullanacağız.

In [7]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=25, patience=5):
    since = time.time()
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_f1 = 0.0
    epochs_no_improve = 0
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}

    # 16-bit (Yarı Hassasiyet) Eğitimi için Scaler tanımlıyoruz (HIZLANDIRICI 🚀)
    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Her epoch'ta training ve validation fazı
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Modeli eğitim moduna al
                dataloader = train_loader
            else:
                model.eval()   # Modeli değerlendirme moduna al (Dropout vb. kapanır)
                dataloader = val_loader

            running_loss = 0.0
            all_preds = []
            all_labels = []

            # Batch'ler üzerinde dön
            for inputs, labels in tqdm(dataloader, desc=f"{phase.capitalize()} Batch"):
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                # Gradient'leri sıfırla
                optimizer.zero_grad()

                # İleri yayılım (Forward pass)
                with torch.set_grad_enabled(phase == 'train'):
                    # AMP Aktif: İşlemleri Tensor Çekirdeklerinde 16-bit hızında yapar
                    with torch.cuda.amp.autocast(enabled=(phase == 'train')):
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)
                    
                    _, preds = torch.max(outputs, 1)

                    # Eğer eğitimdeysek Geri yayılım (Backward pass) ve Optimizasyon
                    if phase == 'train':
                        # Normal loss.backward() yerine Scaler kullanıyoruz
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                running_loss += loss.item() * inputs.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
            if phase == 'train':
                scheduler.step()

            # Metrikleri hesapla
            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = accuracy_score(all_labels, all_preds)
            epoch_f1 = f1_score(all_labels, all_preds, average='macro') # Dengesiz veriler için Macro F1

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} Macro F1: {epoch_f1:.4f}')
            
            # Geçmişi kaydet
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc)
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc)
                history['val_f1'].append(epoch_f1)

                # Early Stopping Kontrolü
                if epoch_f1 > best_f1:
                    best_f1 = epoch_f1
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                    # Modeli diske kaydet
                    torch.save(model.state_dict(), 'best_vit_model.pth')
                else:
                    epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f'Early stopping triggered after {patience} epochs without F1 improvement.')
            break

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Macro F1: {best_f1:4f}')

    # En iyi model ağırlıklarını yükle
    model.load_state_dict(best_model_wts)
    return model, history


In [8]:
# Modeli Eğitme İşlemini Başlat
best_model, history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS, patience=5)


C:\Users\Asus\AppData\Local\Temp\ipykernel_11536\1510599660.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1/30
----------


Train Batch:   0%|          | 0/2819 [00:00<?, ?it/s]C:\Users\Asus\AppData\Local\Temp\ipykernel_11536\1510599660.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(phase == 'train')):
Train Batch: 100%|██████████| 2819/2819 [11:12<00:00,  4.19it/s]


Train Loss: 2.0333 Acc: 0.6406 Macro F1: 0.6378


Val Batch:   0%|          | 0/470 [00:00<?, ?it/s]C:\Users\Asus\AppData\Local\Temp\ipykernel_11536\1510599660.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(phase == 'train')):
Val Batch: 100%|██████████| 470/470 [01:48<00:00,  4.34it/s]


...

Val Loss: 1.7446 Acc: 0.7504 Macro F1: 0.6914

Epoch 30/30
----------


Train Batch:   0%|          | 0/2819 [00:00<?, ?it/s]C:\Users\Asus\AppData\Local\Temp\ipykernel_11536\1510599660.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(phase == 'train')):
Train Batch: 100%|██████████| 2819/2819 [12:37<00:00,  3.72it/s]


Train Loss: 0.8016 Acc: 0.9952 Macro F1: 0.9951


Val Batch:   0%|          | 0/470 [00:00<?, ?it/s]C:\Users\Asus\AppData\Local\Temp\ipykernel_11536\1510599660.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(phase == 'train')):
Val Batch: 100%|██████████| 470/470 [01:51<00:00,  4.23it/s]

Val Loss: 1.7445 Acc: 0.7505 Macro F1: 0.6909

Training complete in 424m 38s
Best val Macro F1: 0.692763
